In [1]:
from timeatlas import TimeAtlas, HR
import pandas as pd

# Instantiate the TimeAtlas client and get a dataset of street network data for Venice in 1808
ta = TimeAtlas('https://api.timeatlas.eu/v1')
ds = ta.get_dataset_by_slug('venice-1808-streetnetwork')
entity_list = ta.materialize_all_rde_from_dataset_obj(ds)

# instantiating a dataframe with the HRs
hrs = [e for e in entity_list if isinstance(e, HR)]
df = ta.hr_list_to_dataframe(hrs).set_index('uuid')
df['hr_object'] = pd.Series(index=[hr.uuid for hr in hrs], data=hrs)

# Extracting geometry and length information from the HR objects and adding them as new columns in the dataframe
df['geometry'] = df['hr_object'].apply(lambda hr: hr.documents[0].has_geometry[0] if hr else None)
df['meter_length'] = df['length'].apply(lambda x: float(x.replace('m', '')) if x else None)

### Displaying all street that are longer than 100M

In [2]:

import folium
import geopandas as gpd
def display_map(df):
    map = folium.Map(location=[45.433, 12.329], tiles="CartoDB Positron", zoom_start=14.6)
    for _, r in df.iterrows():
            geo_j = gpd.GeoSeries(r["geometry"].geometry).to_json()
            geo_j = folium.GeoJson(data=geo_j)
            folium.Popup(r["NAME"]).add_to(geo_j)
            geo_j.add_to(map)
    return map

display_map(df[df['meter_length'] > 100])

# Displaying all streets that are "Calle"

In [3]:
dfn = df[df.NAME.notna()]
display_map(dfn[dfn.NAME.str.lower().str.contains('calle')])